# ACCESS-IoT · Notebook 2 — reference baselines by access-control approach (E4, E5, E7)
Simple, generic reference implementations of each approach, evaluated on the four benchmark tasks:
- **T1** access decision (PERMIT / LIMITED / DENY / BREAK_GLASS) against the oracle;
- **T2** break-glass adjudication (genuine emergency vs fraudulent claim);
- **T3** delegation risk (will a grant be misused?);
- **T4** learning from delayed, action-dependent feedback (chronological replay; only outcomes of granted requests are revealed).

Approaches: static ABAC · ABAC + static break-glass · RBAC · mined ABAC · CapBAC (with and without audit-triggered revocation) ·
trust-based (beta reputation, replayed) · risk-adaptive · zero-trust re-evaluation · supervised gradient boosting (single request / with history) ·
unsupervised insider detection (Isolation Forest) · delayed-reward neural contextual bandit.

Input: the released builds (Zenodo, DOI 10.5281/zenodo.23838631) in `DATA_DIR/builds/<build>/`; splits, views and history
features are part of the release, so this notebook does not need Notebook 1.
Runtime: a GPU is recommended (XGBoost, neural bandit); without one everything runs on the CPU, much more slowly. High RAM recommended.

**Scoring your own method**: run R0 and R1, build `c = Ctx("main")`, produce one action per request (0 PERMIT, 1 LIMITED,
2 DENY, 3 BREAK_GLASS) and call `t1_metrics(actions, c, c.te)`; for T2 / T3 pass a score per request to `t2_metrics` / `t3_metrics`
(T2 official protocol: the actor-disjoint split of the `hard_fraud` build, as in cell R7b).

In [ ]:
# R0 — settings and shared helpers
import os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
# Data: unzip every ACCESS-IoT-v1.0_<build>.zip from Zenodo (https://doi.org/10.5281/zenodo.23838631) into DATA_DIR,
# which gives DATA_DIR/builds/<build>/requests.parquet, ... . Outputs (tables, figures) go to REPORTS.
# On Colab: from google.colab import drive; drive.mount("/content/drive") and set DATA_DIR to the folder on your Drive.
DATA_DIR = Path(os.environ.get("ACCESS_IOT_DATA", "data"))
REPORTS = Path(os.environ.get("ACCESS_IOT_REPORTS", "reports"))
REL, REP = DATA_DIR / "builds", REPORTS
REP.mkdir(parents=True, exist_ok=True)
assert REL.exists(), f"no builds found in {REL}: download and unzip the Zenodo files first"
# main + difficulty variants (E5) + telemetry ablation (E6a: no telemetry, WUSTL-EHMS-2020 instead of CICIoMT2024)
RUN_BUILDS = [b for b in ["main", "sparse_feedback", "noisy_benign", "hard_fraud", "no_telemetry", "wustl_telemetry"]
              if (REL / b / "features/history.parquet").exists()]
SEEDS = [0, 1, 2]
N_JOBS = min(8, os.cpu_count() or 2)
import xgboost as xgb, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("builds:", RUN_BUILDS, "| device:", DEVICE)
P, L, D, B = 0, 1, 2, 3
DEC = np.array(["PERMIT", "LIMITED", "DENY", "BREAK_GLASS"])
ENC = {s: i for i, s in enumerate(DEC)}
VISIBLE_KINDS = ["request", "pip", "telemetry", "baseline"]
ID_COLS = {"RequestID", "Timestamp", "SessionID", "UserID", "PatientID", "DeviceID", "DelegationID", "DelegatorID", "DelegateeID"}
PRES = {"VitalSigns", "MedicalRecord", "Prescription", "Imaging"}

def load(b):
    d = pd.read_parquet(REL / b / "requests.parquet")
    for c in ("Timestamp", "FeedbackTime"): d[c] = pd.to_datetime(d[c])
    return d

def single_row(d, dic):
    cols = [c for c in dic.loc[dic.Visibility.isin(VISIBLE_KINDS), "Column"] if c in d and c not in ID_COLS]
    X = d[cols].copy()
    X["HourOfDay"] = d.Timestamp.dt.hour; X["HasDelegation"] = d.DelegationID.notna().astype(int)
    X["DeviceDeptMatchesUser"] = (d.DeviceDepartment == d.UserDepartment).astype(int)
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]): X[c] = X[c].astype(object).fillna("NA").astype(str).astype("category")
    return X

class Ctx:
    """Everything a baseline may use for one build (oracle columns are kept apart and used only for evaluation)."""
    def __init__(self, b):
        self.b = b; self.d = load(b); d = self.d
        self.dic = pd.read_csv(REL / b / "data_dictionary.csv"); self.pol = json.load(open(REL / b / "policy.json"))
        self.mask = np.load(REL / b / "views/rl_admissible_mask.npy")
        self.split = d.Split.to_numpy(); self.tr, self.va, self.te = (self.split == "train"), (self.split == "val"), (self.split == "test")
        self.X1 = single_row(d, self.dic)
        H = pd.read_parquet(REL / b / "features/history.parquet"); H.index = self.X1.index; self.H = H
        self.X2 = pd.concat([self.X1, H], axis=1)
        self.ev = pd.read_parquet(REL / b / "views/feedback_events.parquet")
        self.opt = d.OptimalDecision.map(ENC).to_numpy(); self.mal = d.IsMalicious.to_numpy() == 1
        self.grant = (d.RequestType == "DelegationGrant").to_numpy(); self.mis = (d.DelegationOutcome == "Misused").to_numpy()
        self.underdel = d.DelegationID.notna().to_numpy() & ~self.grant
        self.zeta = 1 + d.ResourceSensitivity.to_numpy(float) / 4
        self.day = ((d.Timestamp - d.Timestamp.min().normalize()).dt.days).to_numpy()

In [ ]:
# R1 — metrics for T1-T3 (and per-threat malicious-grant rate for E7)
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

def t1_metrics(a, c, rows):
    a, o, m = a[rows], c.opt[rows], c.mal[rows]; g = np.isin(a, [P, L, B])
    gr, mis, ud = c.grant[rows], c.mis[rows], c.underdel[rows]
    bgctx = c.mask[rows, 3] & (c.d.EmergencyFlag.to_numpy()[rows] == 1)
    r = {"Accuracy": (a == o).mean(), "MacroF1": f1_score(o, a, labels=[P, L, D, B], average="macro", zero_division=0),
         "MGR": g[m].mean(), "LDR": (a[o == P] == D).mean(), "EDR": (a[o == B] == D).mean(),
         "BG-Mis": np.isin(a[bgctx & m], [B, L]).mean() if (bgctx & m).any() else np.nan,
         "DG-Mis": np.isin(a[gr & mis], [P, L]).mean() if (gr & mis).any() else np.nan,
         "DG-CleanRefused": (a[gr & ~mis] == D).mean() if (gr & ~mis).any() else np.nan,
         "DU-Exp": g[ud & m].mean() if (ud & m).any() else np.nan, "DU-Den": (a[ud & ~m & (o != D)] == D).mean(),
         "Envelope violations": int((~c.mask[rows][np.arange(rows.sum()), a]).sum()),
         "Avg reward": immediate_reward(a, o, m, c.zeta[rows]).mean()}
    return {k: round(float(v), 4) for k, v in r.items()}

def per_threat(a, c, rows):
    th = c.d.ThreatType.to_numpy()[rows]; g = np.isin(a[rows], [P, L, B]); m = c.mal[rows]
    return {t: round(float(g[m & (th == t)].mean()), 4) for t in sorted(set(th[m]))}

def t2_metrics(score, c, rows):
    """Break-glass adjudication: emergency claims in break-glass contexts; genuine (oracle BREAK_GLASS) vs fraudulent (malicious)."""
    ctx = rows & c.mask[:, 3] & (c.d.EmergencyFlag.to_numpy() == 1)
    gen, fra = ctx & (c.opt == B), ctx & c.mal
    y = np.r_[np.ones(gen.sum()), np.zeros(fra.sum())]; s = np.r_[score[gen], score[fra]]
    if len(np.unique(y)) < 2: return {}
    thr = np.quantile(score[gen], 0.01)                      # honour 99% of genuine emergencies
    return {"T2 AUROC": round(roc_auc_score(y, s), 4), "T2 fraud granted @99% honoured": round(float((score[fra] >= thr).mean()), 4)}

def t3_metrics(risk, c, rows):
    """Delegation risk at grant time: misused (positive) vs clean grants."""
    g = rows & c.grant
    y = c.mis[g].astype(int)
    if y.min() == y.max(): return {}
    return {"T3 AUROC": round(roc_auc_score(y, risk[g]), 4), "T3 AUPRC": round(average_precision_score(y, risk[g]), 4),
            "T3 base rate": round(float(y.mean()), 4)}

def immediate_reward(a, o, mal, z):
    """reward of the benchmark's RL view (rl_env_spec.json); also the cost-sensitive objective used to tune every threshold"""
    r = np.where(a == o, 1.0, 0.0)
    r = np.where((a == L) & np.isin(o, [P, B]), 0.5, r)
    r = np.where(np.isin(a, [P, B]) & (o == L), -z, r)
    r = np.where(np.isin(a, [P, B]) & (o == D), np.where(mal, -5 * z, -2 * z), r)
    r = np.where((a == L) & (o == D), np.where(mal, -2.5 * z, -z), r)
    r = np.where((a == D) & np.isin(o, [P, L]), -1.0, r)
    r = np.where((a == D) & (o == B), -4.0, r)
    return r

def val_reward(a, c):
    v = c.va; return float(immediate_reward(a[v], c.opt[v], c.mal[v], c.zeta[v]).mean())

def tune(fn, grid, c):
    """pick the parameter tuple with the best average reward on the validation period"""
    best = None
    for g in grid:
        a = fn(*g); f = val_reward(a, c)
        if best is None or f > best[0]: best = (f, g, a)
    return best[1], best[2]

In [ ]:
# R2 — rule-based and policy-based approaches (no learning from labels)
def static_abac(c): return np.where(c.mask[:, 0], P, D)
def static_bg(c): return np.where(c.mask[:, 0], P, np.where(c.mask[:, 3], B, D))

def rbac(c):
    d = c.d; perm = set(map(tuple, c.pol["permissions"]))
    ok = np.array([(r, s, a) in perm for r, s, a in zip(d.UserRole, d.ResourceType, d.RequestedAction)])
    return np.where(c.grant, np.where(c.mask[:, 0], P, D), np.where(ok, P, D))         # roles only: no relationship check

def mined_abac(c):
    """ABAC policy mining: an interpretable decision tree learned from the logged (static) decisions of the training period."""
    from sklearn.tree import DecisionTreeClassifier
    d = c.d
    F = pd.get_dummies(d[["UserRole", "ResourceType", "RequestedAction", "RequestType", "DelegationScope"]].astype(str), dtype=np.uint8)
    F = pd.concat([F, d[["OnCareTeam", "SameDepartment", "PatientResearchConsent", "DelegationDepth", "RootDelegatorMaxDepth"]].reset_index(drop=True),
                   pd.DataFrame({"DeviceDeptMatch": (d.DeviceDepartment == d.UserDepartment).astype(int).to_numpy(),
                                 "HasDelegation": d.DelegationID.notna().astype(int).to_numpy(),
                                 "DepthWithinLimit": (d.DelegationDepth <= d.RootDelegatorMaxDepth).astype(int).to_numpy()})], axis=1)
    tree = DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=0).fit(F[c.tr], (d.ABACDecision[c.tr] == "Permit"))
    perm = tree.predict(F).astype(bool)
    fidelity = float((perm[c.te] == (d.ABACDecision.to_numpy()[c.te] == "Permit")).mean())
    return np.where(perm, P, D), {"mined rules (leaves)": int(tree.get_n_leaves()), "fidelity to logged policy (test)": round(fidelity, 4)}

def zero_trust(c):
    """Per-request re-evaluation of posture, network, authentication and context (fixed rules, no learning)."""
    d = c.d
    q = np.quantile(d.NetAnomaly[c.tr], 0.95) if "NetAnomaly" in d else None
    posture = (d.FirmwareAttestation == "Pass") & (d.DeviceAuthFailures24h == 0) & ((d.NetAnomaly <= q) if q is not None else True)
    network = (d.SourceNetwork != "External") | (d.AccessChannel == "Hospital Workstation")
    auth = d.LoginFailedAttempts == 0
    context = (d.WithinShift == 1) | (d.EmergencyFlag == 1)
    score = (posture.astype(int) + network.astype(int) + auth.astype(int) + context.astype(int)).to_numpy()
    a = np.where(c.mask[:, 0] & (score == 4), P, np.where(c.mask[:, 0] & (score == 3), L, D))
    return np.where(c.mask[:, 3] & (posture & auth).to_numpy(), B, a)

def risk_adaptive(c):
    """Risk = 0.5 anomaly + 0.25 sensitivity + 0.25 context; anomaly from an Isolation Forest fitted on training traffic (no labels)."""
    from sklearn.ensemble import IsolationForest
    num = c.X2.select_dtypes(include=[np.number]).fillna(0).to_numpy(np.float32)
    mu, sd = num[c.tr].mean(0), num[c.tr].std(0) + 1e-6; Z = (num - mu) / sd
    iso = IsolationForest(n_estimators=200, max_samples=8192, random_state=0, n_jobs=-1).fit(Z[c.tr])
    an = -iso.score_samples(Z); lo, hi = np.quantile(an[c.tr], [0.01, 0.99]); an = np.clip((an - lo) / (hi - lo), 0, 1)
    d = c.d
    ctx = ((1 - d.WithinShift) + (d.SourceNetwork == "External") + c.H.u_new_device.to_numpy()).to_numpy() / 3
    risk = 0.5 * an + 0.25 * (d.ResourceSensitivity.to_numpy() - 2) / 2 + 0.25 * ctx
    def pol(t1, t2, tb):
        a = np.where(c.mask[:, 0] & (risk < t1), P, np.where(c.mask[:, 0] & (risk < t2), L, D))
        return np.where(c.mask[:, 3] & (risk < tb), B, a)
    qs = np.quantile(risk[c.tr], [0.5, 0.7, 0.8, 0.9, 0.95, 0.98])
    grid = [(t1, t2, tb) for t1 in qs for t2 in qs if t2 >= t1 for tb in qs]
    g, a = tune(pol, grid, c)
    return a, risk, an

def isolation_insider(c, an):
    """Unsupervised insider detection on behaviour history: refuse the most anomalous requests."""
    def pol(qp, qb):
        a = np.where(c.mask[:, 0] & (an < qp), P, D)
        return np.where(c.mask[:, 3] & (an < qb), B, a)
    qs = np.quantile(an[c.tr], [0.8, 0.9, 0.95, 0.98, 0.99, 1.0])
    g, a = tune(pol, [(x, y) for x in qs for y in qs], c)
    return a

In [ ]:
# R3 — approaches that learn from delayed, action-dependent feedback (T4): CapBAC with revocation, trust-based (beta reputation)
def applies(kind_applies, a_src):
    return np.select([kind_applies == "always", kind_applies == "break_glass_granted", kind_applies == "delegation_approved"],
                     [np.ones(len(a_src), bool), np.isin(a_src, [B, L]), np.isin(a_src, [P, L])], np.isin(a_src, [P, L, B]))

def capbac(c, revoke):
    """Capability tokens: role rights (base) and delegated rights within scope, validity window and depth limit; no relationship check.
    With revoke=True, a principal's delegated capabilities are revoked once misuse evidence about it is revealed - but only
    evidence about requests the mechanism actually granted is ever revealed (replayed chronologically, day by day)."""
    d = c.d; perm = set(map(tuple, c.pol["permissions"]))
    role_ok = np.array([(r, s, a) in perm for r, s, a in zip(d.UserRole, d.ResourceType, d.RequestedAction)])
    deleg_ok = d.DelegationID.notna().to_numpy() & (d.CapabilityCheck == "Pass").to_numpy() & \
        (d.DelegationDepth <= d.RootDelegatorMaxDepth).to_numpy()
    own_ok = (d.UserRole != "Patient").to_numpy() | c.mask[:, 0]
    cap = np.where(c.grant, deleg_ok, (role_ok | deleg_ok) & own_ok)
    a = np.where(cap, P, D)
    if not revoke: return a
    # revocation acts on the principal that misused a delegated capability: its delegated tokens stop working, it can no
    # longer receive or re-issue delegations (tokens expire within hours, so revoking only the misused token would be too late)
    deleg = pd.read_parquet(REL / c.b / "delegations.parquet")[["DelegationID", "DelegateeID"]]
    holder_of = dict(zip(deleg.DelegationID, deleg.DelegateeID))
    rid2row = pd.Series(np.arange(len(d)), index=d.RequestID.to_numpy())
    ev = c.ev[c.ev.FeedbackLabel.isin(["Malicious", "DelegationMisused"]) & c.ev.DelegationID.notna()].copy()
    ev["row"] = rid2row[ev.RequestID.to_numpy()].to_numpy()
    ev["holder"] = np.where(ev.FeedbackLabel == "DelegationMisused", ev.DelegationID.map(holder_of), d.UserID.to_numpy()[ev.row.to_numpy()])
    ev["eday"] = ((ev.FeedbackTime - d.Timestamp.min().normalize()).dt.days).to_numpy()
    user, dlg_user = d.UserID.to_numpy(object), d.DelegateeID.to_numpy(object)
    uses_delegation = d.DelegationID.notna().to_numpy() & ~c.grant
    revoked = set(); days = np.unique(c.day); by_day = {k: np.flatnonzero(c.day == k) for k in days}; ev_by_day = ev.groupby("eday")
    for k in days:
        idx = by_day[k]
        if revoked:
            u, v = pd.Series(user[idx]).isin(revoked).to_numpy(), pd.Series(dlg_user[idx]).isin(revoked).to_numpy()
            hit = ((uses_delegation[idx] | c.grant[idx]) & u) | (c.grant[idx] & v)
            a[idx[hit]] = D
        if k in ev_by_day.groups:                            # evidence revealed today takes effect from tomorrow
            e = ev_by_day.get_group(k)
            ok = applies(np.where(e.FeedbackLabel == "DelegationMisused", "delegation_approved", "request_granted"), a[e.row.to_numpy()])
            revoked.update(e.holder[ok].dropna())
    return a

def trust_beta(c):
    """Trust-based access control: beta reputation (L+1)/(L+M+2) of the requester from revealed outcomes, times device reputation
    (0 once a device compromise is revealed). Outcomes reach the store only if the mechanism's own decision exposed them."""
    d = c.d; rid2row = pd.Series(np.arange(len(d)), index=d.RequestID.to_numpy())
    ev = c.ev.copy(); ev["row"] = rid2row[ev.RequestID.to_numpy()].to_numpy()
    ev["eday"] = ((ev.FeedbackTime - d.Timestamp.min().normalize()).dt.days).to_numpy()
    uid = pd.factorize(d.UserID)[0]; did = pd.factorize(d.DeviceID)[0]; nU, nD = uid.max() + 1, did.max() + 1
    days = np.unique(c.day); by_day = {k: np.flatnonzero(c.day == k) for k in days}; evd = {k: g for k, g in ev.groupby("eday")}
    last_val_day = c.day[c.va].max()
    def replay(tp, tb, upto=None):
        Lc, Mc, Dc = np.zeros(nU), np.zeros(nU), np.zeros(nD); a = np.full(len(d), D); T = np.ones(len(d))
        for k in (days if upto is None else days[days <= upto]):
            idx = by_day[k]
            T[idx] = (Lc[uid[idx]] + 1) / (Lc[uid[idx]] + Mc[uid[idx]] + 2) * (Dc[did[idx]] == 0)
            a[idx] = np.where(c.mask[idx, 0] & (T[idx] >= tp), P, np.where(c.mask[idx, 3] & (T[idx] >= tb), B, D))
            if k in evd:
                e = evd[k]; ok = applies(e.AppliesIf.to_numpy(), a[e.row.to_numpy()]); e = e[ok]
                r = e.row.to_numpy()
                np.add.at(Lc, uid[r[e.FeedbackLabel.to_numpy() == "Legitimate"]], 1)
                np.add.at(Mc, uid[r[e.FeedbackLabel.isin(["Malicious", "DelegationMisused"]).to_numpy()]], 1)
                np.add.at(Dc, did[r[(e.Kind == "device_detect").to_numpy()]], 1)
        return a, T
    # coarse-to-fine search on validation reward (64 + <=24 replays instead of 225); tuning replays stop after the validation period
    seen, best = set(), None
    def search(grid):
        nonlocal best
        for g in grid:
            if g in seen: continue
            seen.add(g); a, _ = replay(*g, upto=last_val_day); f = val_reward(a, c)
            if best is None or f > best[0]: best = (f, g)
    coarse = np.round(np.arange(0.05, 0.751, 0.1), 3)
    search([(tp, tb) for tp in coarse for tb in coarse])
    near = lambda v: [x for x in np.round(v + np.array([-0.05, -0.025, 0, 0.025, 0.05]), 3) if 0.05 <= x <= 0.75]
    search([(tp, tb) for tp in near(best[1][0]) for tb in near(best[1][1])])
    a, T = replay(*best[1])                                   # full chronological replay with the chosen thresholds
    return a, T, best[1]

In [ ]:
# R4 — supervised gradient boosting (GPU): decision model on the oracle, plus misuse scores for T2/T3
def xgb_model(seed, multi):
    p = dict(n_estimators=500, learning_rate=0.06, max_depth=8, subsample=0.9, colsample_bytree=0.8, tree_method="hist",
             device=DEVICE, enable_categorical=True, max_cat_to_onehot=1, random_state=seed)
    return xgb.XGBClassifier(objective="multi:softprob", **p) if multi else xgb.XGBClassifier(**p)

def gbdt(c, X, seed, train_mask=None):
    tr = c.tr if train_mask is None else train_mask
    cls = np.unique(c.opt[tr]); remap = np.searchsorted(cls, c.opt[tr])           # XGBoost needs contiguous labels
    m = xgb_model(seed, len(cls) > 2).fit(X[tr], remap); pp = m.predict_proba(X)
    pr = np.zeros((len(X), 4)); pr[:, cls] = pp if pp.shape[1] == len(cls) else np.c_[1 - pp[:, -1], pp[:, -1]]
    q = np.where(c.mask, pr, -1.0); a = q.argmax(1)                                 # masked arg-max: never outside the envelope
    s = xgb_model(seed, False).fit(X[tr], c.mal[tr].astype(int)).predict_proba(X)[:, 1]
    return a, pr, s

In [ ]:
# R5 — delayed-reward neural contextual bandit (GPU): learns only from the reward signal, rewards of deferred decisions arrive
#      with the revealed outcome (or a no-news reward after H days); chronological replay, masked actions
import torch.nn as nn
REW_NN, H_DAYS = 0.5, 21

def resolved_reward(a, label, z):
    return np.select([np.isin(label, ["Malicious", "DelegationMisused"]), label == "Legitimate", label == "CompromisedPath"],
                     [-5 * z * np.where(a == L, 0.5, 1.0), np.where(np.isin(a, [P, B]), 1.0, 0.5), np.where(a == L, 1.0, -z)], REW_NN)

def bandit(c, X, seed, passes=4, gamma_steps=4):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); d = c.d
    num = X.select_dtypes(include=[np.number]).fillna(0).to_numpy(np.float32)
    cat = pd.get_dummies(X.select_dtypes(exclude=[np.number]).astype(str), dtype=np.float32).to_numpy()
    F = np.c_[num, cat]; mu, sd = F[c.tr].mean(0), F[c.tr].std(0) + 1e-6; F = ((F - mu) / sd).astype(np.float32)
    Ft = torch.tensor(F, device=DEVICE); Mt = torch.tensor(c.mask, device=DEVICE)
    net = nn.Sequential(nn.Linear(F.shape[1], 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 4)).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), 3e-4)
    emerg = d.EmergencyFlag.to_numpy() == 1
    rid2row = pd.Series(np.arange(len(d)), index=d.RequestID.to_numpy())
    fb = c.ev.assign(row=rid2row[c.ev.RequestID.to_numpy()].to_numpy())
    fb = fb.assign(fday=((fb.FeedbackTime - d.Timestamp.min().normalize()).dt.days).to_numpy()).drop_duplicates("row", keep="last")
    # per-row lookup arrays (one revealed event per request at most; rows without an event never resolve before the horizon)
    has_fb = np.zeros(len(d), bool); has_fb[fb.row] = True
    fb_day = np.zeros(len(d), int); fb_day[fb.row] = fb.fday.to_numpy()
    fb_lab = np.full(len(d), "", object); fb_lab[fb.row] = fb.FeedbackLabel.to_numpy()
    fb_app = np.full(len(d), "always", object); fb_app[fb.row] = fb.AppliesIf.to_numpy()
    train_days = np.unique(c.day[c.tr]); by_day = {k: np.flatnonzero((c.day == k) & c.tr) for k in train_days}
    def q_all(rows):
        with torch.no_grad(): return torch.where(Mt[rows], net(Ft[rows]), torch.tensor(-1e9, device=DEVICE)).cpu().numpy()
    best = (-1, None)
    for ps in range(passes):
        eps = max(0.05, 1.0 - ps / max(passes - 1, 1))
        buf_s, buf_a, buf_r = [], [], []; pending = {}          # due_day -> [(rows, actions, labels)]
        for k in train_days:
            idx = by_day[k]
            q = q_all(idx); a = q.argmax(1)
            rnd = rng.random(len(idx)) < eps
            if rnd.any():                                       # uniform random admissible action (vectorised)
                a[rnd] = (rng.random((rnd.sum(), 4)) * c.mask[idx[rnd]]).argmax(1)
            defr = (a == B) | ((a == L) & emerg[idx]) | (c.grant[idx] & np.isin(a, [P, L]))
            r_imm = immediate_reward(a, c.opt[idx], c.mal[idx], c.zeta[idx])
            buf_s.append(idx[~defr]); buf_a.append(a[~defr]); buf_r.append(r_imm[~defr])
            if defr.any():
                di, da = idx[defr], a[defr]
                okf = has_fb[di] & (fb_day[di] <= k + H_DAYS) & applies(fb_app[di], da)
                due_day = np.where(okf, fb_day[di], k + H_DAYS); lab = np.where(okf, fb_lab[di], "")
                for dd in np.unique(due_day):
                    s_ = due_day == dd; pending.setdefault(int(dd), []).append((di[s_], da[s_], lab[s_]))
            due = [x for dd in sorted(dd for dd in pending if dd <= k) for x in pending.pop(dd)]
            if due:
                rr = np.concatenate([x[0] for x in due]); aa = np.concatenate([x[1] for x in due]); lab = np.concatenate([x[2] for x in due])
                buf_s.append(rr); buf_a.append(aa); buf_r.append(resolved_reward(aa, lab, c.zeta[rr]))
            S = np.concatenate(buf_s[-60:]); A = np.concatenate(buf_a[-60:]); R = np.concatenate(buf_r[-60:])
            for _ in range(gamma_steps):
                j = rng.integers(0, len(S), min(2048, len(S)))
                qv = net(Ft[S[j]]).gather(1, torch.tensor(A[j], device=DEVICE)[:, None]).squeeze(1)
                loss = nn.functional.smooth_l1_loss(qv, torch.tensor(R[j], dtype=torch.float32, device=DEVICE))
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 10); opt.step()
        av = q_all(np.flatnonzero(c.va)).argmax(1); a_full = np.full(len(d), D); a_full[c.va] = av
        f = val_reward(a_full, c)
        if f > best[0]: best = (f, {k: v.clone() for k, v in net.state_dict().items()})
    net.load_state_dict(best[1])
    q = q_all(np.arange(len(d)))
    return q.argmax(1), q

In [ ]:
# R6 — run every approach on every build. Checkpointed: each finished build is saved to reports/E4_parts/<build>.pkl, so if the
#      runtime disconnects, re-run R0-R5 and this cell; finished builds are loaded, not recomputed. Delete a .pkl to redo that build.
ALL, THREAT, T23, EXTRA = [], [], [], {}
PART = REP / "E4_parts"; PART.mkdir(parents=True, exist_ok=True)
for b in RUN_BUILDS:
    pf = PART / f"{b}.pkl"
    if pf.exists():
        k = pd.read_pickle(pf); ALL += k["ALL"]; THREAT += k["THREAT"]; T23 += k["T23"]; EXTRA.update(k["EXTRA"])
        print(f"{b}: loaded from checkpoint"); continue
    n0 = (len(ALL), len(THREAT), len(T23)); x0 = set(EXTRA)
    t0 = time.time(); c = Ctx(b); te = c.te
    res = {}
    res["Static ABAC"] = (static_abac(c), None, None)
    res["ABAC + static break-glass"] = (static_bg(c), None, None)
    res["RBAC"] = (rbac(c), None, None)
    a, info = mined_abac(c); res["Mined ABAC (policy mining)"] = (a, None, None); EXTRA[(b, "mined")] = info
    res["CapBAC"] = (capbac(c, False), None, None)
    res["CapBAC + audit revocation"] = (capbac(c, True), None, None)
    a, T, th = trust_beta(c); res["Trust-based (beta reputation)"] = (a, T, 1 - T); EXTRA[(b, "trust thresholds")] = th
    a, risk, an = risk_adaptive(c); res["Risk-adaptive"] = (a, -risk, risk)
    res["Zero-trust re-evaluation"] = (zero_trust(c), None, None)
    res["Insider detection (Isolation Forest)"] = (isolation_insider(c, an), -an, an)
    print(f"{b}: rule/feedback approaches done ({time.time()-t0:.0f}s)")
    for nm, X in [("Supervised GBDT, single request", c.X1), ("Supervised GBDT, with history", c.X2)]:
        for sd in SEEDS:
            a, pr, s = gbdt(c, X, sd)
            res[f"{nm}#{sd}"] = (a, pr[:, B] - pr[:, D], pr[:, D])     # T3: probability that the oracle refuses the grant
    print(f"{b}: gradient boosting done ({time.time()-t0:.0f}s)")
    for sd in SEEDS:
        a, q = bandit(c, c.X2, sd)
        res[f"Delayed-reward neural bandit#{sd}"] = (a, q[:, B] - q[:, D], q[:, D] - np.maximum(q[:, P], q[:, L]))
        print(f"{b}: bandit seed {sd} done ({time.time()-t0:.0f}s)")
    for nm, (a, s2, s3) in res.items():
        base, sd = (nm.split("#") + [None])[:2]
        ALL.append({"build": b, "method": base, "seed": sd, **t1_metrics(a, c, te)})
        THREAT.append({"build": b, "method": base, "seed": sd, **per_threat(a, c, te)})
        T23.append({"build": b, "method": base, "seed": sd, **(t2_metrics(s2, c, te) if s2 is not None else {}),
                    **(t3_metrics(s3, c, te) if s3 is not None else {})})
    np.savez_compressed(REP / f"E4_decisions_{b}.npz", **{k.replace("#", "_s"): v[0] for k, v in res.items()})
    pd.to_pickle({"ALL": ALL[n0[0]:], "THREAT": THREAT[n0[1]:], "T23": T23[n0[2]:],
                  "EXTRA": {k: v for k, v in EXTRA.items() if k not in x0}}, pf)
    del c, res; import gc; gc.collect(); torch.cuda.empty_cache()
    print(f"{b}: saved checkpoint ({time.time()-t0:.0f}s)")

def agg(rows):
    t = pd.DataFrame(rows); g = t.drop(columns="seed").groupby(["build", "method"], sort=False)
    m, s = g.mean(), g.std().fillna(0)
    return m, s
T1m, T1s = agg(ALL); THm, _ = agg(THREAT); T23m, T23s = agg(T23)
T1m.round(4).to_csv(REP / "E4_T1_mean.csv"); T1s.round(4).to_csv(REP / "E4_T1_sd.csv")
THm.round(4).to_csv(REP / "E7_per_threat_MGR.csv"); T23m.round(4).to_csv(REP / "E4_T2_T3_mean.csv")
json.dump({f"{k[0]}|{k[1]}": v for k, v in EXTRA.items()}, open(REP / "E4_extra.json", "w"), indent=1, default=str)
print(T1m.loc["main"].round(3).to_string()); print(T23m.loc["main"].round(3).to_string())

In [ ]:
# R7 — generalisation splits for the strongest supervised baseline: leave-one-threat-out and department shift (main build)
c = Ctx("main"); sp = pd.read_parquet(REL / "main/splits.parquet"); rows = []
for col in [x for x in sp.columns if x.startswith("loto_")]:
    t = col[5:]; trm = (sp[col] == "train").to_numpy()
    a, _, _ = gbdt(c, c.X2, 0, trm)
    th = c.d.ThreatType.to_numpy(); g = np.isin(a, [P, L, B])
    rows.append({"split": f"held out: {t}", "held-out threat MGR (test)": round(float(g[c.te & c.mal & (th == t)].mean()), 4),
                 "seen-in-training MGR": round(float(THm.loc[("main", "Supervised GBDT, with history"), t]), 4) if t in THm else np.nan})
trm, tem = (sp.dept_shift == "train").to_numpy(), (sp.dept_shift == "test").to_numpy()
a, _, _ = gbdt(c, c.X2, 0, trm)
rows.append({"split": "department shift", **{f"T1 {k}": v for k, v in t1_metrics(a, c, tem).items() if k in ("MacroF1", "MGR", "LDR")}})
gen = pd.DataFrame(rows); gen.to_csv(REP / "E4_generalisation.csv", index=False); print(gen.to_string(index=False))

In [ ]:
# R7b — T2 on the official actor-disjoint split (held-out fraudsters), with bootstrap 95% intervals.
#        Learned methods are retrained without any fraud by the held-out actors; label-free methods are re-tuned on the same
#        validation rows. On hard_fraud the result is also broken down into overt and covert fraudsters.
def t2_ci(score, g, f, boot=1000):
    sg, sf = score[g], score[f]
    if len(sg) == 0 or len(sf) == 0: return {}
    auc = roc_auc_score(np.r_[np.ones(len(sg)), np.zeros(len(sf))], np.r_[sg, sf])
    rng = np.random.default_rng(0); bs = []
    for _ in range(boot):
        a, b_ = rng.choice(sg, len(sg)), rng.choice(sf, len(sf))
        bs.append(roc_auc_score(np.r_[np.ones(len(a)), np.zeros(len(b_))], np.r_[a, b_]))
    lo, hi = np.quantile(bs, [0.025, 0.975])
    return {"T2 AUROC": round(auc, 3), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
            "fraud granted @99% honoured": round(float((sf >= np.quantile(sg, 0.01)).mean()), 3), "genuine": len(sg), "fraud": len(sf)}

T2ROWS = []
for b in [x for x in ["main", "hard_fraud"] if x in RUN_BUILDS]:
    c = Ctx(b); sp = pd.read_parquet(REL / b / "splits.parquet")
    c.tr, c.va = (sp.actor_disjoint == "train").to_numpy(), (sp.actor_disjoint == "val").to_numpy()
    tem = (sp.actor_disjoint == "test").to_numpy()
    ctx = tem & c.mask[:, 3] & (c.d.EmergencyFlag.to_numpy() == 1)
    gen, fra = ctx & (c.opt == B), ctx & c.mal & (c.d.ThreatType == "emergency_misuse").to_numpy()
    tag = c.d.ActorTag.to_numpy()
    scores = {}
    _, T, _ = trust_beta(c); scores["Trust-based (beta reputation)"] = T
    _, risk, an = risk_adaptive(c); scores["Risk-adaptive"] = -risk; scores["Insider detection (Isolation Forest)"] = -an
    _, pr, _ = gbdt(c, c.X2, 0); scores["Supervised GBDT, with history"] = pr[:, B] - pr[:, D]
    _, q = bandit(c, c.X2, 0); scores["Delayed-reward neural bandit"] = q[:, B] - q[:, D]
    for m, s in scores.items():
        T2ROWS.append({"build": b, "method": m, "fraudsters": "all held-out", **t2_ci(s, gen, fra)})
        if b == "hard_fraud":
            for t_, nm in [("EmergencyMisuser", "overt"), ("CovertEmergencyMisuser", "covert")]:
                T2ROWS.append({"build": b, "method": m, "fraudsters": nm, **t2_ci(s, gen, fra & (tag == t_))})
    print(f"{b}: T2 actor-disjoint done")
t2tab = pd.DataFrame(T2ROWS); t2tab.to_csv(REP / "E4_T2_actor_disjoint.csv", index=False); print(t2tab.to_string(index=False))

In [ ]:
# R8 — Table 5 draft (main build + variants) and Fig. 3 (per-threat difficulty heatmap)
import matplotlib.pyplot as plt
order = list(T1m.loc["main"].index)
tab = T1m.loc["main"][["MacroF1", "MGR", "LDR", "EDR", "BG-Mis", "DG-Mis", "DU-Exp", "Envelope violations"]].copy()
for v in [x for x in RUN_BUILDS if x != "main"]:
    tab[f"MGR ({v})"] = T1m.loc[v]["MGR"].reindex(order)
tab = tab.join(T23m.loc["main"][[x for x in ["T2 AUROC", "T2 fraud granted @99% honoured", "T3 AUROC"] if x in T23m]])
tab.round(3).to_csv(REP / "table5_baselines.csv"); print(tab.round(3).to_string())
hm = THm.loc["main"].reindex(order)
fig, ax = plt.subplots(figsize=(8, 0.35 * len(hm) + 1.5))
im = ax.imshow(hm.to_numpy(float), cmap="Reds", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels([x.replace("_", " ") for x in hm.columns], rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(len(hm))); ax.set_yticklabels(hm.index, fontsize=8)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        v = hm.iat[i, j]; ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if v > 0.6 else "black")
fig.colorbar(im, ax=ax, label="malicious-grant rate (test)", shrink=0.8); fig.tight_layout()
fig.savefig(REP / "fig3_per_threat.pdf"); fig.savefig(REP / "fig3_per_threat.png", dpi=200); plt.show()
print("NEXT: Notebook 3 (telemetry swap, synthetic <-> semi-synthetic transfer, explanation ground truth)")